# 06 · Panel-Level Exploration (Bonus)

**Goal:** Explore whether embedding individual *panels* rather than full *pages* produces meaningfully better retrieval.

**Hypothesis:** A page of 6 panels covers multiple scenes and characters. Embedding the whole page averages out these signals. A panel-level embedding is more focused — a query for *"a character looking surprised"* should return a single panel, not a page where that moment is one of six.

**Namespace:** `ns-panels`

**Panel detection approach:** OpenCV contour detection. Comic panels have strong rectangular borders; edge detection + bounding box extraction works surprisingly well on classic golden-age comics without needing an ML model.

---

## When to use panel-level embeddings
- Maximum retrieval precision (the story-builder demo becomes much more compelling)
- Character-level queries: "find all panels featuring a woman in a cape"
- Panel count is manageable — 5 comics × 30 pages × ~6 panels = ~900 panels

In [ ]:
import json
import os
from pathlib import Path

import cv2
import numpy as np
import voyageai
from dotenv import load_dotenv
from IPython.display import display
from PIL import Image
from pinecone import Pinecone
from tqdm import tqdm

load_dotenv()

TRANSCRIPTS_DIR = Path("data/transcripts")
PANELS_DIR      = Path("data/panels")
PANELS_DIR.mkdir(parents=True, exist_ok=True)

INDEX_NAME  = "comics-multimodal"
NAMESPACE   = "ns-panels"
MIN_PANEL_W = 100   # pixels — ignore tiny boxes (captions, gutters)
MIN_PANEL_H = 100

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
vo = voyageai.Client()

info  = pc.preview.indexes.describe(name=INDEX_NAME)
index = pc.index(host=info.host)
print(f"Connected to {INDEX_NAME}")

## Panel detection with OpenCV

Classic comics have clean rectangular panel borders. We:
1. Convert to grayscale and apply binary threshold
2. Find external contours
3. Filter by minimum area and aspect ratio
4. Return bounding boxes sorted top-to-bottom, left-to-right (reading order)

In [ ]:
def detect_panels(image_path: str, min_w: int = MIN_PANEL_W, min_h: int = MIN_PANEL_H) -> list[tuple]:
    """Return list of (x, y, w, h) bounding boxes for detected panels."""
    img = cv2.imread(image_path)
    if img is None:
        return []
    gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw  = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    bw     = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    h_img, w_img = img.shape[:2]
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        # Skip boxes that are the whole page or too small
        if w < min_w or h < min_h:
            continue
        if w > w_img * 0.95 and h > h_img * 0.95:
            continue
        boxes.append((x, y, w, h))

    # Sort reading order: top-to-bottom, then left-to-right
    boxes.sort(key=lambda b: (b[1] // (h_img // 4), b[0]))
    return boxes


def crop_panel(image_path: str, box: tuple, out_path: Path) -> str | None:
    """Crop a panel bounding box and save. Returns path or None on failure."""
    if out_path.exists():
        return str(out_path)
    img = Image.open(image_path).convert("RGB")
    x, y, w, h = box
    panel = img.crop((x, y, x + w, y + h))
    panel.save(out_path, quality=90)
    return str(out_path)

In [ ]:
# Detect and crop all panels, build panel manifest
records    = [json.loads(p.read_text()) for p in sorted(TRANSCRIPTS_DIR.glob("*.json"))]
panel_data = []

for r in tqdm(records, desc="Detecting panels"):
    boxes = detect_panels(r["file_path"])
    if not boxes:
        # Fall back: treat the whole page as a single panel
        boxes = [(0, 0, 9999, 9999)]
    slug_dir = PANELS_DIR / r["comic_slug"]
    slug_dir.mkdir(parents=True, exist_ok=True)
    for i, box in enumerate(boxes):
        panel_id = f"{r['page_id']}_panel{i+1:02d}"
        out_path = slug_dir / f"{panel_id}.jpg"
        if box == (0, 0, 9999, 9999):
            # Copy full page
            import shutil
            shutil.copy(r["file_path"], out_path)
        else:
            crop_panel(r["file_path"], box, out_path)
        panel_data.append({
            "panel_id":    panel_id,
            "page_id":     r["page_id"],
            "comic_title": r["comic_title"],
            "comic_slug":  r["comic_slug"],
            "page_num":    r["page_num"],
            "panel_num":   i + 1,
            "file_path":   str(out_path),
        })

print(f"Total panels detected: {len(panel_data)}  ({len(panel_data)/len(records):.1f} per page avg)")

## Spot-check panel crops

In [ ]:
# Show the first 6 panels from the first page
first_page_panels = [p for p in panel_data if p["page_id"] == panel_data[0]["page_id"]][:6]
fig, axes = plt.subplots(1, len(first_page_panels), figsize=(14, 4))
if len(first_page_panels) == 1:
    axes = [axes]
for ax, p in zip(axes, first_page_panels):
    ax.imshow(Image.open(p["file_path"]))
    ax.set_title(f"Panel {p['panel_num']}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Detected panels — {first_page_panels[0]['page_id']}", fontsize=11)
plt.tight_layout()
plt.show()

## Embed panels and upsert to ns-panels

In [ ]:
import matplotlib.pyplot as plt

EMBED_BATCH = 10

for i in tqdm(range(0, len(panel_data), EMBED_BATCH), desc="Embedding panels"):
    batch = panel_data[i : i + EMBED_BATCH]
    needs = [p for p in batch if not p.get("image_vector")]
    if not needs:
        continue
    inputs  = [[Image.open(p["file_path"]).convert("RGB")] for p in needs]
    result  = vo.multimodal_embed(inputs, model="voyage-multimodal-3", input_type="document")
    for p, vec in zip(needs, result.embeddings):
        p["image_vector"] = vec

print("Panel vectors ready.")

In [ ]:
documents = [
    {
        "_id":        p["panel_id"],
        "embedding":  p["image_vector"],
        "comic_title": p["comic_title"],
        "page_id":    p["page_id"],
        "page_num":   p["page_num"],
        "panel_num":  p["panel_num"],
        "file_path":  p["file_path"],
    }
    for p in panel_data if p.get("image_vector")
]

index.documents.batch_upsert(
    namespace=NAMESPACE,
    documents=documents,
    batch_size=50,
    show_progress=True,
)
print(f"Upserted {len(documents)} panels to {NAMESPACE}.")

## Compare: page-level vs panel-level retrieval

Run the same query against `ns-image-clip` (page-level) and `ns-panels` (panel-level) and see which gives more precise results.

In [ ]:
def compare_page_vs_panel(query: str, top_k: int = 4):
    qvec = vo.multimodal_embed([[query]], model="voyage-multimodal-3", input_type="query").embeddings[0]

    resp_page = index.documents.search(
        namespace="ns-image-clip",
        top_k=top_k,
        score_by=[{"type": "dense_vector", "field": "embedding", "query": qvec}],
        include_fields=["page_id", "comic_title", "page_num", "file_path"],
    )
    resp_panel = index.documents.search(
        namespace=NAMESPACE,
        top_k=top_k,
        score_by=[{"type": "dense_vector", "field": "embedding", "query": qvec}],
        include_fields=["page_id", "comic_title", "page_num", "panel_num", "file_path"],
    )

    fig, axes = plt.subplots(2, top_k, figsize=(14, 8))
    fig.suptitle(f'Query: "{query}"', fontsize=12, fontweight="bold")

    for col, m in enumerate(resp_page.matches):
        ax = axes[0][col]
        fp = getattr(m, "file_path", None)
        if fp and Path(fp).exists():
            ax.imshow(Image.open(fp))
        ax.set_title(f"Page-level\n{getattr(m, 'comic_title', '')} p{getattr(m, 'page_num', '')}", fontsize=8)
        ax.axis("off")

    for col, m in enumerate(resp_panel.matches):
        ax = axes[1][col]
        fp = getattr(m, "file_path", None)
        if fp and Path(fp).exists():
            ax.imshow(Image.open(fp))
        panel_n = getattr(m, "panel_num", "")
        ax.set_title(f"Panel-level\n{getattr(m, 'comic_title', '')} p{getattr(m, 'page_num', '')} panel{panel_n}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
compare_page_vs_panel("character looking surprised or shocked")

In [ ]:
compare_page_vs_panel("close up face of a villain sneering")

In [ ]:
compare_page_vs_panel("two people talking at a table")